Note: Due to GPU runtime and compute constraints, the 30-seed robustness experiments were executed in three batches. This notebook contains the consolidated 30-seed configuration used for the final robustness analysis.

In [ ]:
# =========================
# Setup: paths, seeds, and caption CSV loading
# =========================

import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch

from urllib.parse import unquote

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Not running in Colab, skipping drive.mount().")


# =========================
# Paths
# =========================

# Dataset root
BASE_DIR = "/content/drive/Shareddrives/data255/data255"

# Fine-tuned Qwen split CSVs
SPLITS_ROOT = "/content/drive/MyDrive/data255_revision_2/model/preprocessing/qwen_splits_from_llava_canonical"

# Standalone Qwen caption CSV
CAPTION_CSV = "/content/drive/MyDrive/data255_revision_2/model/preprocessing/qwen25vl_caption_full.csv"

# Fine-tuned image-only CLIP checkpoints
CLIP_FINETUNED_ROOT = "/content/drive/Shareddrives/data255/data255/Fine-tuned CLIP models"

# Output root for 30-seed gated fine-tuned experiment
EXPERIMENT_ROOT = (
    "/content/drive/MyDrive/data255_revision_2/model/"
    "gated_fine_tuned/result/finetuned_clip_qwen_gated_llava_canonical_30seeds"
)


# =========================
# Toggles / hyperparameters
# =========================

RUN_ALL_SEEDS = True
SMOKE_SEEDS = [13]  # used when RUN_ALL_SEEDS=False and for path audit

LEARNING_RATE = 5e-5
BATCH_SIZE = 32
EARLY_STOPPING_PATIENCE = 5
DROPOUT = 0.5
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 20
MODEL_INIT_SEED = 42
NUM_WORKERS = 0


# =========================
# Device
# =========================

def pick_training_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def print_torch_compute_diagnostics():
    print("torch:", torch.__version__)
    print("torch.version.cuda:", getattr(torch.version, "cuda", None))
    print("torch.cuda.is_available():", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA device 0:", torch.cuda.get_device_name(0))


device = pick_training_device()
print_torch_compute_diagnostics()
print("Selected device:", device)


# =========================
# Seed discovery
# =========================

def discover_split_seeds(splits_root: str):
    global SPLIT_DIR_BY_SEED

    SPLIT_DIR_BY_SEED = {}
    required = ["train.csv", "val.csv", "test.csv"]

    for name in os.listdir(splits_root):
        full = os.path.join(splits_root, name)
        if not os.path.isdir(full):
            continue

        if not all(os.path.isfile(os.path.join(full, f)) for f in required):
            continue

        # supports seed_13, split_seed_13, 13, etc.
        m = re.search(r"(\d+)$", name)
        if not m:
            print("Skipping split folder with no seed number:", name)
            continue

        seed = int(m.group(1))
        SPLIT_DIR_BY_SEED[seed] = full

    return sorted(SPLIT_DIR_BY_SEED)


# =========================
# Image path resolver
# =========================

def resolve_split_image_path(row_image_path, base_dir):
    base = base_dir or "."
    p = str(row_image_path)

    if not os.path.isabs(p):
        rel = p.replace(chr(92), "/")
        dataset_top = os.path.join(base, "dataset")

        # If split path starts with dataset/ but actual files are under data/raw dataset/
        if rel.startswith("dataset/") and not os.path.isdir(dataset_top):
            p = os.path.join(base, "data", "raw dataset", rel[len("dataset/"):])
        else:
            p = os.path.join(base, p)

    if "%" in p:
        parts = p.replace(chr(92), "/").split("/")
        p = "/".join(unquote(part) if "%" in part else part for part in parts)

    return os.path.normpath(p)


# =========================
# Caption CSV loading
# =========================

def load_qwen_captions(csv_path: str, base_dir: str):
    df = pd.read_csv(csv_path)

    if "status" in df.columns:
        df = df[df["status"].astype(str).str.lower() == "success"].copy()

    if "caption" not in df.columns:
        raise ValueError("Expected column 'caption' in caption CSV")

    if "image_path" not in df.columns:
        raise ValueError("Expected column 'image_path' in caption CSV")

    captions = {}

    def register(raw_path, caption_text):
        if not isinstance(caption_text, str) or not caption_text.strip():
            return

        cap = caption_text.strip()
        raw = str(raw_path)

        # raw path
        captions[raw] = cap

        # normalized raw path
        captions[os.path.normpath(raw)] = cap

        # absolute raw path joined with BASE_DIR
        if base_dir and not os.path.isabs(raw):
            abs_raw = os.path.normpath(os.path.join(base_dir, raw))
            captions[abs_raw] = cap

        # resolved path using same resolver as split dataset
        resolved = resolve_split_image_path(raw, base_dir)
        captions[resolved] = cap

    for _, row in df.iterrows():
        register(row["image_path"], row["caption"])

    return captions, df


# =========================
# Path sanity checks
# =========================

if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(f"BASE_DIR missing: {BASE_DIR}")

if not os.path.isdir(SPLITS_ROOT):
    raise FileNotFoundError(f"SPLITS_ROOT missing: {SPLITS_ROOT}")

if not os.path.isfile(CAPTION_CSV):
    raise FileNotFoundError(f"CAPTION_CSV missing: {CAPTION_CSV}")

if not os.path.isdir(CLIP_FINETUNED_ROOT):
    raise FileNotFoundError(f"CLIP_FINETUNED_ROOT missing: {CLIP_FINETUNED_ROOT}")


# =========================
# Load seeds / captions / classes
# =========================

SEEDS = discover_split_seeds(SPLITS_ROOT)

captions_dict, caption_df = load_qwen_captions(CAPTION_CSV, BASE_DIR)
print("Qwen caption dict size:", len(captions_dict))

if "style" not in caption_df.columns:
    raise ValueError("Qwen caption CSV must have 'style' column for num_classes")

all_styles = sorted(caption_df["style"].dropna().astype(str).unique())
style_to_idx = {s: i for i, s in enumerate(all_styles)}
num_classes = len(all_styles)


# =========================
# Output folders
# =========================

METRICS_DIR = os.path.join(EXPERIMENT_ROOT, "metrics")
ARTIFACTS_DIR = os.path.join(EXPERIMENT_ROOT, "artifacts")

for d in [
    METRICS_DIR,
    os.path.join(METRICS_DIR, "experiments"),
    os.path.join(ARTIFACTS_DIR, "models"),
    os.path.join(ARTIFACTS_DIR, "learning_curves"),
    os.path.join(ARTIFACTS_DIR, "comparison_plots"),
]:
    os.makedirs(d, exist_ok=True)


print("BASE_DIR:", BASE_DIR)
print("SPLITS_ROOT:", SPLITS_ROOT)
print("CAPTION_CSV:", CAPTION_CSV)
print("CLIP_FINETUNED_ROOT:", CLIP_FINETUNED_ROOT)
print("EXPERIMENT_ROOT:", EXPERIMENT_ROOT)
print("num_classes:", num_classes)
print("SEEDS count:", len(SEEDS), "| first few:", SEEDS[:5])


with open(os.path.join(METRICS_DIR, "seeds_list.txt"), "w") as f:
    f.write("Seeds from SPLITS_ROOT/seed_*\n")
    for s in SEEDS:
        f.write(f"{s}\n")


# =========================
# Path audit
# =========================

def print_fusion_path_audit(smoke_seed: int):
    print("=== Path audit required files / dirs ===")

    img_ok = (
        os.path.isdir(os.path.join(BASE_DIR, "dataset"))
        or os.path.isdir(os.path.join(BASE_DIR, "data", "raw dataset"))
    )

    items = [
        ("BASE_DIR", os.path.isdir(BASE_DIR)),
        ("dataset/ OR data/raw dataset/", img_ok),
        ("SPLITS_ROOT", os.path.isdir(SPLITS_ROOT)),
        ("CAPTION_CSV", os.path.isfile(CAPTION_CSV)),
        ("CLIP_FINETUNED_ROOT", os.path.isdir(CLIP_FINETUNED_ROOT)),
    ]

    for label, ok in items:
        print(f"  [{'x' if ok else ' '}] {label}")

    if not all(ok for _, ok in items):
        raise RuntimeError("Path audit failed — fix paths above.")

    #train_csv = os.path.join(SPLITS_ROOT, f"seed_{smoke_seed}", "train.csv")
    if "SPLIT_DIR_BY_SEED" in globals() and smoke_seed in SPLIT_DIR_BY_SEED:
        smoke_split_dir = SPLIT_DIR_BY_SEED[smoke_seed]
    else:
        smoke_split_dir = os.path.join(SPLITS_ROOT, f"seed_{smoke_seed}")

    train_csv = os.path.join(smoke_split_dir, "train.csv")

    if not os.path.isfile(train_csv):
        raise FileNotFoundError(f"Missing split train.csv: {train_csv}")

    tr = pd.read_csv(train_csv, nrows=5)

    if "image_path" not in tr.columns:
        raise ValueError(f"Missing image_path column in {train_csv}")

    ok_files = 0
    ok_caps = 0

    for i, row in tr.iterrows():
        raw = str(row["image_path"])
        resolved = resolve_split_image_path(raw, BASE_DIR)

        image_exists = os.path.isfile(resolved)
        ok_files += int(image_exists)

        cap_exists = any(
            k in captions_dict
            for k in [
                raw,
                os.path.normpath(raw),
                resolved,
                os.path.normpath(resolved),
            ]
        )
        ok_caps += int(cap_exists)

        print(f"  [{'x' if image_exists else ' '}] smoke row {i} image on disk")
        print(f"      [{'x' if cap_exists else ' '}] Qwen caption key")
        print(f"      raw={raw}")
        print(f"      resolved={resolved}")

    if ok_files == 0:
        raise RuntimeError("Path audit: sample rows resolve to no image files.")

    if ok_caps == 0:
        raise RuntimeError("Path audit: sample rows match no Qwen captions.")

    ckpt_candidates = [
        os.path.join(CLIP_FINETUNED_ROOT, f"seed_{smoke_seed}_best_model.pth"),
        os.path.join(CLIP_FINETUNED_ROOT, f"seed_{smoke_seed}", "best_model.pth"),
        os.path.join(CLIP_FINETUNED_ROOT, f"seed_{smoke_seed}", "clip_model.pth"),
    ]

    ckpt_found = None
    for p in ckpt_candidates:
        if os.path.isfile(p):
            ckpt_found = p
            break

    if ckpt_found is None:
        print("Tried checkpoint paths:")
        for p in ckpt_candidates:
            print("  ", p)
        raise FileNotFoundError(f"Missing CLIP checkpoint for smoke seed {smoke_seed}")

    print(f"  [x] smoke CLIP checkpoint: {ckpt_found}")
    print("=== Path audit OK ===")


print_fusion_path_audit(SMOKE_SEEDS[0])

In [ ]:
import time
import warnings
from datetime import datetime
from pathlib import Path

import torch.nn as nn
import matplotlib.pyplot as plt

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

In [ ]:
# --- toggles ---
RUN_ALL_SEEDS = True
SMOKE_SEEDS = [13]  # used when RUN_ALL_SEEDS=False and for path audit

# --- training hyperparameters ---
LEARNING_RATE = 5e-5
BATCH_SIZE = 32
EARLY_STOPPING_PATIENCE = 5
DROPOUT = 0.5
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 20
MODEL_INIT_SEED = 42
NUM_WORKERS = 0


def pick_training_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def print_torch_compute_diagnostics():
    print("torch:", torch.__version__)
    vcuda = getattr(torch.version, "cuda", None)
    print("torch.version.cuda (wheel build):", vcuda)
    print("torch.cuda.is_available():", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA device 0:", torch.cuda.get_device_name(0))
        return


device = pick_training_device()
print_torch_compute_diagnostics()
print("Selected device:", device)

print("RUN_ALL_SEEDS =", RUN_ALL_SEEDS)
print("Seeds to run =", SEEDS if RUN_ALL_SEEDS else SMOKE_SEEDS)
print("n =", len(SEEDS if RUN_ALL_SEEDS else SMOKE_SEEDS))

In [ ]:
# --- sanity checks ---
if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(f"BASE_DIR missing: {BASE_DIR}")

if not os.path.isdir(SPLITS_ROOT):
    raise FileNotFoundError(f"SPLITS_ROOT missing: {SPLITS_ROOT}")

if not os.path.isdir(CLIP_FINETUNED_ROOT):
    raise FileNotFoundError(f"CLIP_FINETUNED_ROOT missing: {CLIP_FINETUNED_ROOT}")

if not os.path.isfile(CAPTION_CSV):
    print(f"WARNING: Caption CSV missing: {CAPTION_CSV}")
    print("Caption CSV not found. Check CAPTION_CSV or configure caption loading from split CSVs.")

## Load CLIP + BERT

In [ ]:
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
import clip
from transformers import AutoTokenizer, AutoModel

print("Loading CLIP ViT-B/32 …")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device, jit=False)
clip_model = clip_model.float()
clip_model.eval()

print("Loading BERT …")
fashionbert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
fashionbert_model = AutoModel.from_pretrained("bert-base-uncased").to(device)
fashionbert_model.eval()


def load_finetuned_clip_weights(clip_model, ckpt_path, map_location):
    # Load partial fine-tuned CLIP from ImageOnlyFashionClassifier checkpoint.
    ckpt_path = Path(ckpt_path)
    if not ckpt_path.is_file():
        raise FileNotFoundError(f"Missing checkpoint: {ckpt_path}")
    sd = torch.load(ckpt_path, map_location=map_location)
    clip_sd = {k[len("clip_model.") :]: v for k, v in sd.items() if k.startswith("clip_model.")}
    if len(clip_sd) == 0:
        raise ValueError(f"No clip_model.* keys in {ckpt_path} (wrong checkpoint format?)")
    clip_model.load_state_dict(clip_sd, strict=True)
    clip_model.eval()
    for p in clip_model.parameters():
        p.requires_grad = False
    print("  Loaded fine-tuned CLIP:", ckpt_path.name, "|", len(clip_sd), "tensor keys")


print(" Encoders ready (reload CLIP weights per seed before training).")


## Dataset + gated fusion model

In [ ]:
class FashionMultiModalDataset(Dataset):
    # Split CSV rows + Qwen captions + CLIP preprocess. Excludes rows without file or caption.

    def _resolve_path(self, row_image_path):
        return resolve_split_image_path(str(row_image_path), self.base_dir or ".")

    def _caption_lookup(self, resolved, raw):
        for k in (resolved, raw, os.path.normpath(str(raw))):
            if k in self.captions_dict:
                return self.captions_dict[k]
        return None

    def __init__(self, df, captions_dict, style_to_idx, clip_preprocess, base_dir=None):
        self.df = df.reset_index(drop=True)
        self.captions_dict = captions_dict
        self.style_to_idx = style_to_idx
        self.clip_preprocess = clip_preprocess
        self.base_dir = base_dir

        self.valid_indices = []
        missing_file = 0
        missing_cap = 0
        for idx in range(len(self.df)):
            row = self.df.iloc[idx]
            raw = str(row["image_path"])
            resolved = self._resolve_path(raw)
            cap = self._caption_lookup(resolved, raw)
            if cap is None:
                missing_cap += 1
                continue
            if not os.path.isfile(resolved):
                missing_file += 1
                continue
            self.valid_indices.append(idx)

        print(
            f"  Dataset: {len(self.valid_indices)} / {len(self.df)} usable | missing caption: {missing_cap} | missing file: {missing_file}"
        )

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        row = self.df.iloc[actual_idx]
        raw_key = str(row["image_path"])
        image_path = self._resolve_path(raw_key)
        if not os.path.isfile(image_path):
            raise FileNotFoundError(f"Missing image after filter: {image_path}")
        image = Image.open(image_path).convert("RGB")
        image = self.clip_preprocess(image)
        caption = self._caption_lookup(image_path, raw_key)
        if caption is None:
            raise RuntimeError(f"Missing caption for {image_path}")
        style = row["style"]
        label = self.style_to_idx[str(style)]
        return {
            "image": image,
            "caption": caption,
            "label": label,
            "style": style,
            "image_path": image_path,
        }


class GatedFusion(nn.Module):
    def __init__(self, visual_dim=512, textual_dim=768):
        super().__init__()
        self.gate_generator = nn.Sequential(
            nn.Linear(visual_dim + textual_dim, 1),
            nn.Sigmoid()
        )
        self.text_projection = nn.Linear(textual_dim, visual_dim)

    def forward(self, visual_features, textual_features):
        combined = torch.cat([visual_features, textual_features], dim=-1)
        g = self.gate_generator(combined)
        text_projected = self.text_projection(textual_features)
        fused = g * visual_features + (1.0 - g) * text_projected
        return fused


class MultiModalFashionClassifier(nn.Module):
    def __init__(self, clip_model, fashionbert_model, num_classes, dropout=0.5, visual_dim=512, textual_dim=768):
        super().__init__()
        self.clip_model = clip_model
        self.fashionbert_model = fashionbert_model
        for p in self.clip_model.parameters():
            p.requires_grad = False
        for p in self.fashionbert_model.parameters():
            p.requires_grad = False
        self.fusion = GatedFusion(visual_dim, textual_dim)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def train(self, mode=True):
        super().train(mode)
        self.clip_model.eval()
        self.fashionbert_model.eval()
        return self

    def forward(self, images, captions):
        with torch.no_grad():
            visual_features = self.clip_model.encode_image(images).float()
            inputs = fashionbert_tokenizer(
                list(captions),
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=128,
            ).to(images.device)
            outputs = self.fashionbert_model(**inputs)
            textual_features = outputs.last_hidden_state[:, 0, :]
        fused = self.fusion(visual_features, textual_features)
        return self.classifier(fused)


def train_epoch(model, train_loader, criterion, optimizer, dev):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for batch in tqdm(train_loader, desc="train", leave=False):
        images = batch["image"].to(dev)
        captions = batch["caption"]
        labels = batch["label"].to(dev)
        optimizer.zero_grad()
        logits = model(images, captions)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pred = logits.argmax(dim=1)
        total += labels.size(0)
        correct += (pred == labels).sum().item()
    acc = correct / max(total, 1)
    return total_loss / max(len(train_loader), 1), acc


def validate_epoch(model, val_loader, criterion, dev, collect_meta=False):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_predictions, all_labels = [], []
    all_paths, all_styles = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="val", leave=False):
            images = batch["image"].to(dev)
            captions = batch["caption"]
            labels = batch["label"].to(dev)
            logits = model(images, captions)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            pred = logits.argmax(dim=1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
            all_predictions.extend(pred.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
            if collect_meta:
                all_paths.extend(list(batch["image_path"]))
                all_styles.extend(list(batch["style"]))
    macro = f1_score(all_labels, all_predictions, average="macro", zero_division=0) if all_predictions else 0.0
    acc = correct / max(total, 1)
    if collect_meta:
        return total_loss / max(len(val_loader), 1), acc, all_predictions, all_labels, macro, all_paths, all_styles
    return total_loss / max(len(val_loader), 1), acc, all_predictions, all_labels, macro


print(" Dataset + model classes defined")
ATTN_FUSION_DATASET_CELL_VERSION = 3


## Runner: load splits, reload CLIP per seed, train

In [ ]:
def load_split_csvs(seed: int):
    if "SPLIT_DIR_BY_SEED" in globals() and seed in SPLIT_DIR_BY_SEED:
        split_dir = SPLIT_DIR_BY_SEED[seed]
    else:
        split_dir = os.path.join(SPLITS_ROOT, f"seed_{seed}")

    if not os.path.isdir(split_dir):
        raise FileNotFoundError(f"Missing split_dir for seed {seed}: {split_dir}")

    train_path = os.path.join(split_dir, "train.csv")
    val_path = os.path.join(split_dir, "val.csv")
    test_path = os.path.join(split_dir, "test.csv")

    for p in [train_path, val_path, test_path]:
        if not os.path.isfile(p):
            raise FileNotFoundError(f"Missing split csv: {p}")

    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)
    test_df = pd.read_csv(test_path)

    return split_dir, train_df, val_df, test_df


def make_seed_worker(base_seed):
    def _fn(worker_id):
        w = int(base_seed) + int(worker_id)
        np.random.seed(w)
        torch.manual_seed(w)

    return _fn


def run_robustness_experiment(seed_value, seed_idx):
    _exp_ds_ver = 3
    if globals().get("ATTN_FUSION_DATASET_CELL_VERSION") != _exp_ds_ver:
        raise RuntimeError(
            "Dataset cell is missing or out of date (FashionMultiModalDataset / path remap). "
            "Use **Kernel -> Restart**, then **Run All** from the first cell.\n"
            f"Expected ATTN_FUSION_DATASET_CELL_VERSION == {_exp_ds_ver!r}, "
            f"got {globals().get('ATTN_FUSION_DATASET_CELL_VERSION')!r}."
        )
    print(f"\n{'='*70}\nExperiment {seed_idx}/{len(SEEDS)}: seed {seed_value}\n{'='*70}")

    ckpt_path = Path(CLIP_FINETUNED_ROOT) / f"seed_{seed_value}_best_model.pth"
    assert str(seed_value) in ckpt_path.name

    result_file = os.path.join(METRICS_DIR, "experiments", f"seed_{seed_value}_results.json")
    if os.path.exists(result_file):
        print("  ⏭️  Result exists, skipping")
        with open(result_file) as f:
            return json.load(f)

    split_dir, train_df, val_df, test_df = load_split_csvs(seed_value)

    print("  Split sizes:", len(train_df), len(val_df), len(test_df))
    for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
        ds_tmp = FashionMultiModalDataset(part, captions_dict, style_to_idx, clip_preprocess, base_dir=BASE_DIR)
        print(f"    {name}: with captions+image {len(ds_tmp)} / {len(part)}")

    load_finetuned_clip_weights(clip_model, ckpt_path, map_location=device)

    torch.manual_seed(MODEL_INIT_SEED)
    np.random.seed(MODEL_INIT_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(MODEL_INIT_SEED)

    train_ds = FashionMultiModalDataset(train_df, captions_dict, style_to_idx, clip_preprocess, base_dir=BASE_DIR)
    val_ds = FashionMultiModalDataset(val_df, captions_dict, style_to_idx, clip_preprocess, base_dir=BASE_DIR)
    test_ds = FashionMultiModalDataset(test_df, captions_dict, style_to_idx, clip_preprocess, base_dir=BASE_DIR)

    if len(train_ds) == 0:
        r0 = str(train_df.iloc[0]["image_path"]) if len(train_df) else "<empty train_df>"
        res0 = resolve_split_image_path(r0, BASE_DIR) if len(train_df) else ""
        cap0 = (
            any(k in captions_dict for k in (res0, r0, os.path.normpath(str(r0))))
            if len(train_df)
            else False
        )
        file0 = os.path.isfile(res0) if len(train_df) else False
        if len(train_df) and file0 and cap0:
            raise RuntimeError(
                "Training dataset is empty even though the first train row resolves to an existing file "
                "and has a Qwen caption. The FashionMultiModalDataset class in memory is almost certainly "
                "out of date (old image-path logic).\n\n"
                "Fix: Jupyter menu **Kernel -> Restart**, then **Run All** from the first cell so the "
                "Dataset class and resolve_split_image_path stay in sync.\n"
                f"(Debug: resolved first row -> {res0!r})"
            )
        raise RuntimeError(
            "Training dataset is empty (0 rows with Qwen caption + existing image). Common causes: "
            "stale kernel after notebook edits, wrong PROJECT_ROOT, or caption/path mismatch.\n"
            f"  PROJECT_ROOT={PROJECT_ROOT}\n"
            f"  BASE_DIR={BASE_DIR}\n"
            f"  first train image_path={r0!r}\n"
            f"  resolved={res0!r} exists={os.path.isfile(res0)}\n"
            f"  dataset/ at repo root: {os.path.isdir(os.path.join(BASE_DIR, 'dataset'))}\n"
            f"  data/raw dataset/: {os.path.isdir(os.path.join(BASE_DIR, 'data', 'raw dataset'))}"
        )

    loader_seed = MODEL_INIT_SEED + int(seed_value)
    g = torch.Generator()
    g.manual_seed(loader_seed)
    try:
        train_loader = DataLoader(
            train_ds,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            worker_init_fn=make_seed_worker(loader_seed) if NUM_WORKERS > 0 else None,
            generator=g,
        )
    except ValueError as e:
        if "num_samples" in str(e):
            raise RuntimeError(
                "Training DataLoader failed because the training set has length 0 (RandomSampler). "
                "If you updated this notebook, use **Kernel -> Restart** then **Run All** from the top "
                "so the FashionMultiModalDataset definition matches the setup cell (resolve_split_image_path / "
                "data/raw dataset remap).\n"
                f"len(train_ds)={len(train_ds)!r}\n"
                f"Original error: {e!r}"
            ) from e
        raise
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    train_valid_df = train_ds.df.iloc[train_ds.valid_indices]
    class_weights = compute_class_weight(
        "balanced",
        classes=np.arange(num_classes),
        y=train_valid_df["style"].map(style_to_idx).values,
    )
    class_weights = torch.FloatTensor(class_weights).to(device)

    model = MultiModalFashionClassifier(clip_model, fashionbert_model, num_classes=num_classes, dropout=DROPOUT).to(device)
    n_trainable = sum(p.numel() for p in model.fusion.parameters()) + sum(p.numel() for p in model.classifier.parameters())
    print("  Trainable parameters (fusion + classifier only):", n_trainable)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(
        list(model.fusion.parameters()) + list(model.classifier.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    trainable_names = [n for n, p in model.named_parameters() if p.requires_grad]
    print("  Trainable parameter names (first 25):", trainable_names[:25])
    print("  Total trainable tensors:", len(trainable_names))
    assert all(
        n.startswith("fusion.") or n.startswith("classifier.") for n in trainable_names
    ), f"Unexpected trainable params: {trainable_names}"

    fusion_head_path = os.path.join(ARTIFACTS_DIR, "models", f"seed_{seed_value}_best_fusion_head.pth")

    train_losses, val_losses, train_accs, val_accs, val_macro_f1s, learning_rates = [], [], [], [], [], []
    best_val_macro_f1 = -1.0
    best_epoch = 0
    patience_counter = 0
    best_val_loss = float("inf")
    early_stopped = False
    start_time = time.time()

    for epoch in range(MAX_EPOCHS):
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        va_loss, va_acc, _, _, va_f1 = validate_epoch(model, val_loader, criterion, device)
        scheduler.step()
        learning_rates.append(scheduler.get_last_lr()[0])
        train_losses.append(tr_loss)
        val_losses.append(va_loss)
        train_accs.append(tr_acc)
        val_accs.append(va_acc)
        val_macro_f1s.append(va_f1)

        if va_f1 > best_val_macro_f1:
            best_val_macro_f1 = va_f1
            best_epoch = epoch + 1
            patience_counter = 0
            torch.save(
                {
                    "fusion": model.fusion.state_dict(),
                    "classifier": model.classifier.state_dict(),
                    "seed": int(seed_value),
                    "split_mode": "csv",
                    "clip_checkpoint": str(ckpt_path),
                    "caption_csv": CAPTION_CSV,
                    "best_epoch": int(best_epoch),
                    "best_val_macro_f1": float(best_val_macro_f1),
                },
                fusion_head_path,
            )
        else:
            patience_counter += 1
        if va_loss < best_val_loss:
            best_val_loss = va_loss
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            early_stopped = True
            print(f"  Early stop at epoch {epoch+1}")
            break
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1}: tr_loss={tr_loss:.4f} val_loss={va_loss:.4f} val_macro_f1={va_f1:.4f}")

    total_time = time.time() - start_time
    if not os.path.exists(fusion_head_path):
        torch.save(
            {
                "fusion": model.fusion.state_dict(),
                "classifier": model.classifier.state_dict(),
                "seed": int(seed_value),
                "split_mode": "csv",
                "clip_checkpoint": str(ckpt_path),
                "caption_csv": CAPTION_CSV,
                "best_epoch": int(best_epoch),
                "best_val_macro_f1": float(best_val_macro_f1),
                "fallback_save": True,
            },
            fusion_head_path,
        )
        print("  Warning: no fusion-head checkpoint during training; saved fallback from last weights.")

    ck = torch.load(fusion_head_path, map_location=device)
    model.fusion.load_state_dict(ck["fusion"])
    model.classifier.load_state_dict(ck["classifier"])
    model.eval()
    te_loss, te_acc, te_pred, te_lab, te_f1, te_paths, te_styles = validate_epoch(
        model, test_loader, criterion, device, collect_meta=True
    )

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes[0, 0].plot(train_losses, label="train")
    axes[0, 0].plot(val_losses, label="val")
    axes[0, 0].legend()
    axes[0, 1].plot(train_accs, label="train acc")
    axes[0, 1].plot(val_accs, label="val acc")
    axes[0, 1].set_ylabel("accuracy (0–1)")
    axes[0, 1].legend()
    axes[1, 0].plot(val_macro_f1s, label="val macro F1")
    axes[1, 0].legend()
    axes[1, 1].axis("off")
    axes[1, 1].text(
        0.05,
        0.5,
        f"seed={seed_value}\nbest_val_macro_f1={best_val_macro_f1:.4f}\ntest_macro_f1={te_f1:.4f}\ntest_acc={te_acc:.4f}",
        fontsize=11,
        family="monospace",
    )
    plt.suptitle(
        f"Gated fusion + Qwen — seed {seed_value} (CSV splits; fine-tuned CLIP)"
    )
    plt.tight_layout()
    plt.savefig(os.path.join(ARTIFACTS_DIR, "learning_curves", f"seed_{seed_value}_learning_curves.png"), dpi=200, bbox_inches="tight")
    plt.close()

    results = {
        "experiment_id": f"seed_{seed_value}",
        "seed_value": seed_value,
        "seed_index": seed_idx,
        "timestamp": datetime.now().isoformat(),
        "protocol": {
            "split_mode": "csv",
            "splits": "data/splits/seed_<N>/{train,val,test}.csv",
            "clip_checkpoint": str(ckpt_path),
            "caption_csv": CAPTION_CSV,
            "legacy_table_csv": None,
            "fusion": "gated",
            "fusion_head_checkpoint": fusion_head_path,
        },
        "configuration": {
            "learning_rate": float(LEARNING_RATE),
            "batch_size": BATCH_SIZE,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "dropout": DROPOUT,
            "weight_decay": float(WEIGHT_DECAY),
            "max_epochs": MAX_EPOCHS,
            "model_init_seed": MODEL_INIT_SEED,
        },
        "validation_metrics": {
            "best_val_macro_f1": float(best_val_macro_f1),
            "best_epoch": int(best_epoch),
            "best_val_accuracy": float(val_accs[best_epoch - 1]) if best_epoch > 0 else 0.0,
        },
        "test_metrics": {
            "test_macro_f1": float(te_f1),
            "test_accuracy": float(te_acc),
            "test_loss": float(te_loss),
            "test_predictions": [int(x) for x in te_pred],
            "test_labels": [int(x) for x in te_lab],
            "test_image_paths": [str(x) for x in te_paths],
            "test_styles": [str(x) for x in te_styles],
        },
        "caption_coverage": {
            "train_rows_csv": int(len(train_df)),
            "val_rows_csv": int(len(val_df)),
            "test_rows_csv": int(len(test_df)),
            "train_with_caption_and_file": int(len(train_ds)),
            "val_with_caption_and_file": int(len(val_ds)),
            "test_with_caption_and_file": int(len(test_ds)),
        },
        "training_info": {
            "total_epochs": len(train_losses),
            "early_stopped": early_stopped,
            "total_time_minutes": float(total_time / 60.0),
        },
        "data_split_info": {
            "split_dir": split_dir,
            "train_size": len(train_df),
            "val_size": len(val_df),
            "test_size": len(test_df),
        },
    }
    out_json = os.path.join(METRICS_DIR, "experiments", f"seed_{seed_value}_results.json")
    with open(out_json, "w") as f:
        json.dump(results, f, indent=2)
    print(f"  Done seed {seed_value} | best val macro F1={best_val_macro_f1:.4f} | test macro F1={te_f1:.4f}")
    return results


if RUN_ALL_SEEDS:
    all_results = []
    failed = []
    for i, sv in enumerate(SEEDS, 1):
        try:
            all_results.append(run_robustness_experiment(sv, i))
        except Exception as e:
            print(f"FAIL seed {sv}: {e}")
            failed.append((sv, str(e)))
    print("Completed:", len(all_results), "Failed:", len(failed))
else:
    for i, sv in enumerate(SMOKE_SEEDS, 1):
        run_robustness_experiment(sv, i)
